In [41]:
import pandas as pd

# Using my previous dataset and converting it into dataframe

df = pd.DataFrame({'content': [
    "WOW!!! This movie is SOOOO good 😍",
    "I bought this product for 999 rupees!!!",
    "mail us at test@gmail.com for any query.",
    "Contact me at test@gmail.com ASAP.",
    "<p>This movie was AMAZING!</p>",
    "I am soooo happy with this service!!!",
    "u r gr8 bro",
    "The product is GOOD... but delivery was SLOW.",
    "I waited 2 HOURS for my order!!!",
    "This is    a very    good product.",
    "OMG!!! This phone is absolutely AMAZING!!! 🔥",
    "The price is only 1499 and the quality is excellent.",    
    "Check www.example.com for the latest offers.",    
    "Please email support@company.com for help.",    
    "<div>This product is worth buying.</div>",    
    "The delivery was reallllllly fast!!!",    
    "I loooooove this product ❤️",    
    "idk why the delivery was so late :(",    
    "This movie was kinda boring but the acting was GREAT.",    
    "The service was bad!!! Very bad!!!",    
    "I ordered it yesterday at 10:30 PM.",    
    "     The product arrived in perfect condition.     ",    
    "This product is 100% genuine & reliable.",    
    "u should definitely buy this product!!!",    
    "The camera quality is sooo sooo good 😍",    
    "I am very happy with my purchase :)",    
    "This app is AWESOME!!! Download it now!!!",    
    "The customer support team was helpful... but response was slow.",    
    "OMG!!! Why is this product soooo expensive??? 😡",    
    "gr8 product, fast delivery, totally worth it!!!"
]
})

In [42]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import  word_tokenize
from nltk.stem import WordNetLemmatizer

# Using my previous codes to clean text to get final_clean_text 

en_stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
slangs_dict = {
    "u": "you",
    "r": "are",
    "gr8": "great",
    "idk": "i don't know",
    "asap": "as soon as possible",
    "omg": "oh my god",
    "bro": "brother"
}

def final_nlp_pipeline(text):

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Removing punctuation 
    text = re.sub(r'[!"#$%&\'()*+,-;=?[\\\]^_`{|}~]', '', text)

    # Remove special characters and emojis
    text = re.sub(r'[^\w\s]', '', text)
    
    # Normalize repeated characters
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Expand Slangs
    words = text.split()
    expanded_words = [slangs_dict.get(word, word) for word in words]

    # Stop word removal
    filtered_words = []
    for word in expanded_words:
        if word not in en_stop_words:
            filtered_words.append(word)

    # Tokenization
    tokens = word_tokenize(" ".join(filtered_words))

    # Lemmatization
    lemmatized_words = []
    for word in tokens:
        lemmatized_words.append(lemmatizer.lemmatize(word))

    return " ".join(lemmatized_words)

df["final_processed_text"] = df["content"].apply(final_nlp_pipeline)
df[["content", "final_processed_text"]].head(10)

,content,final_processed_text
0,WOW!!! This movie is SOOOO good 😍,wow movie soo good
1,I bought this product for 999 rupees!!!,bought product rupee
2,mail us at test@gmail.com for any query.,mail u query
3,Contact me at test@gmail.com ASAP.,contact a soon a possible
4,<p>This movie was AMAZING!</p>,movie amazing
5,I am soooo happy with this service!!!,soo happy service
6,u r gr8 bro,gr brother
7,The product is GOOD... but delivery was SLOW.,product good delivery slow
8,I waited 2 HOURS for my order!!!,waited hour order
9,This is a very good product.,good product


## Task 1 -> Manual One Hot Encoding

In [43]:
# Selected 5 sentences from the cleaned text
sentences = df['final_processed_text'].tail(5).tolist()
print("Sentences:")
print(sentences)

# Vocabulary from the selected sentences
vocabulary = sorted(set(word for sentence in sentences for word in sentence.split()))
print("\nVocabulary:")
print(vocabulary)

# One Hot Vectors
one_hot_vectors = []
for sentence in sentences:
    words = sentence.split()
    vector = []
    for word in vocabulary:
        if word in words:
            vector.append(1)
        else:
            vector.append(0)
    one_hot_vectors.append(vector)
            
# For Displaying
for sentence, vector in zip(sentences, one_hot_vectors):
    print(f"\nSentence: {sentence}")
    print(f"Vector: {vector}")


Sentences:
['happy purchase', 'app awesome download', 'customer support team helpful response slow', 'oh my god product soo expensive', 'gr product fast delivery totally worth']

Vocabulary:
['app', 'awesome', 'customer', 'delivery', 'download', 'expensive', 'fast', 'god', 'gr', 'happy', 'helpful', 'my', 'oh', 'product', 'purchase', 'response', 'slow', 'soo', 'support', 'team', 'totally', 'worth']

Sentence: happy purchase
Vector: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]

Sentence: app awesome download
Vector: [1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Sentence: customer support team helpful response slow
Vector: [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0]

Sentence: oh my god product soo expensive
Vector: [0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0]

Sentence: gr product fast delivery totally worth
Vector: [0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1]


## Task 2 -> One-Hot Encoding Using Scikit-learn

In [44]:
from sklearn.feature_extraction.text import CountVectorizer

# Create CountVectorizer
cv = CountVectorizer(binary=True)

# Fit and transform the selected 5 sentences
one_hot_matrix = cv.fit_transform(sentences)

# Printing one hot vocabulary
print(cv.get_feature_names_out())

# Display One-Hot matrix
print("One-Hot Encoded Matrix:")
print(one_hot_matrix.toarray())

['app' 'awesome' 'customer' 'delivery' 'download' 'expensive' 'fast' 'god'
 'gr' 'happy' 'helpful' 'my' 'oh' 'product' 'purchase' 'response' 'slow'
 'soo' 'support' 'team' 'totally' 'worth']
One-Hot Encoded Matrix:
[[0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0]
 [1 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 1 1 0 1 1 0 0]
 [0 0 0 0 0 1 0 1 0 0 0 1 1 1 0 0 0 1 0 0 0 0]
 [0 0 0 1 0 0 1 0 1 0 0 0 0 1 0 0 0 0 0 0 1 1]]


## Task 3 -> Bag of Words Representation

In [48]:
# Create BoW vectorizer
bow_vectorizer = CountVectorizer()

# Fit and transform the cleaned dataset
bow_matrix = bow_vectorizer.fit_transform(df["final_processed_text"])

# Get vocabulary
bow_vocabulary = bow_vectorizer.get_feature_names_out()

# Display vocabulary size
print("Vocabulary Size:")
print(len(bow_vocabulary))

# Display sample vocabulary
print("\nSample Vocabulary:")
print(bow_vocabulary[:20])

# Display sample feature vectors
print("\nSample BoW Feature Vectors:")
print(bow_matrix[:5].toarray())

Vocabulary Size:
71

Sample Vocabulary:
['absolutely' 'acting' 'amazing' 'app' 'arrived' 'awesome' 'bad' 'boring'
 'bought' 'brother' 'buy' 'buying' 'camera' 'check' 'condition' 'contact'
 'customer' 'definitely' 'delivery' 'do']

Sample BoW Feature Vectors:
[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0
  0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]


## Task 4 -> Understanding Word Frequency

In [ ]:
import numpy as np

# Calculate total frequency of every word
word_frequency = np.asarray(bow_matrix.sum(axis=0)).flatten()

frequency_df = pd.DataFrame({
    "word": bow_vocabulary,
    "frequency": word_frequency
})

# Top 10 Most Frequent Words
print("Top 10 Most Frequent Words:")
print(frequency_df.sort_values(by='frequency',ascending=False).head(10))

print("\nLeast Frequent Words:")
print(frequency_df.sort_values(by='frequency',ascending=False).tail(10))

# BoW captures frequency by storing the number of times each vocabulary word appears in each document. 
# Therefore, repeated words receive higher count values while words that do not appear receive a value of zero.

Top 10 Most Frequent Words:
        word  frequency
52   product         10
62       soo          5
18  delivery          4
27      good          4
40     movie          3
6        bad          2
2    amazing          2
41        my          2
60   service          2
61      slow          2

Least Frequent Words:
         word  frequency
56     really          1
55      query          1
59      rupee          1
63       soon          1
65       team          1
64    support          1
66    totally          1
67     waited          1
69        wow          1
70  yesterday          1


## Task 5 -> Unigrams, Bigrams & Trigrams

In [72]:
# Unigram
unigram_vectorizer = CountVectorizer(ngram_range=(1,1))
unigram_matrix = unigram_vectorizer.fit_transform(df['final_processed_text'])
unigram_vocabulary = unigram_vectorizer.get_feature_names_out()

print("Unigram Vocabulary Size:")
print(len(unigram_vocabulary))

print("Sample Unigram Features:")
print(unigram_vocabulary[:20])

print("Sample Unigram Vectors:")
print(unigram_matrix[:5].toarray())

# Bigram
bigram_vectorizer = CountVectorizer(ngram_range=(2,2))
bigram_matrix = bigram_vectorizer.fit_transform(df['final_processed_text'])
bigram_vocabulary = bigram_vectorizer.get_feature_names_out()

print("\nBigram Vocabulary Size:")
print(len(bigram_vocabulary))

print("Sample Bigram Features:")
print(bigram_vocabulary[:20])

print("Sample Bigram Vectors:")
print(bigram_matrix[:5].toarray())

# Trigram
trigram_vectorizer = CountVectorizer(ngram_range=(3,3))
trigram_matrix = trigram_vectorizer.fit_transform(df['final_processed_text'])
trigram_vocabulary = trigram_vectorizer.get_feature_names_out()

print("\nTrigram Vocabulary Size:")
print(len(trigram_vocabulary))

print("Sample Trigram Features:")
print(trigram_vocabulary[:20])

print("Sample Trigram Vectors:")
print(trigram_matrix[:5].toarray())


Unigram Vocabulary Size:
71
Sample Unigram Features:
['absolutely' 'acting' 'amazing' 'app' 'arrived' 'awesome' 'bad' 'boring'
 'bought' 'brother' 'buy' 'buying' 'camera' 'check' 'condition' 'contact'
 'customer' 'definitely' 'delivery' 'do']
Sample Unigram Vectors:
[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0
  0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

In [74]:
# Comparing Unigram, Bigram and Trigram

ngram_comparison = pd.DataFrame({
    "N-Gram": [
        "Unigram",
        "Bigram",
        "Trigram"
    ],
    
    "Vocabulary Size": [
        len(unigram_vocabulary),
        len(bigram_vocabulary),
        len(trigram_vocabulary)
    ]
})

print(ngram_comparison)

print("\nSample Unigrams:")
print(unigram_vocabulary[:10])

print("\nSample Bigrams:")
print(bigram_vocabulary[:10])

print("\nSample Trigrams:")
print(trigram_vocabulary[:10])

    N-Gram  Vocabulary Size
0  Unigram               71
1   Bigram               71
2  Trigram               43

Sample Unigrams:
['absolutely' 'acting' 'amazing' 'app' 'arrived' 'awesome' 'bad' 'boring'
 'bought' 'brother']

Sample Bigrams:
['absolutely amazing' 'acting great' 'app awesome' 'arrived perfect'
 'awesome download' 'bad bad' 'boring acting' 'bought product'
 'buy product' 'camera quality']

Sample Trigrams:
['app awesome download' 'arrived perfect condition' 'boring acting great'
 'bought product rupee' 'camera quality soo' 'check latest offer'
 'contact soon possible' 'customer support team' 'definitely buy product'
 'delivery really fast']


## Task 6 -> Combined N-Grams

In [75]:
combined_vectorizer = CountVectorizer(ngram_range=(1, 2))

combined_matrix = combined_vectorizer.fit_transform(df['final_processed_text'])

combined_vocabulary = combined_vectorizer.get_feature_names_out()

print("Combined N-Gram Vocabulary Size:")
print(len(combined_vocabulary))

print("\nSample Combined Features:")
print(combined_vocabulary[:30])

print("\nSample Combined N-Gram Vectors:")
print(combined_matrix[:5].toarray())

Combined N-Gram Vocabulary Size:
142

Sample Combined Features:
['absolutely' 'absolutely amazing' 'acting' 'acting great' 'amazing' 'app'
 'app awesome' 'arrived' 'arrived perfect' 'awesome' 'awesome download'
 'bad' 'bad bad' 'boring' 'boring acting' 'bought' 'bought product'
 'brother' 'buy' 'buy product' 'buying' 'camera' 'camera quality' 'check'
 'check latest' 'condition' 'contact' 'contact soon' 'customer'
 'customer support']

Sample Combined N-Gram Vectors:
[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0
  0 0 0 0 0 0 

In [77]:
# Comparing it with unigram and combined gram in which we found both unigram and bigram
comparison = pd.DataFrame({
    "Vectorization": [
        "Unigram",
        "Unigram + Bigram"
    ],
    
    "Vocabulary Size": [
        len(unigram_vocabulary),
        len(combined_vocabulary)
    ]
})

comparison

,Vectorization,Vocabulary Size
0,Unigram,71
1,Unigram + Bigram,142


## Task 7 -> TF-IDF Implementation

In [83]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer()

# Fit and transform the entire dataset
tfidf_matrix = tfidf.fit_transform(df['final_processed_text'])

tfidf_vocabulary = tfidf.get_feature_names_out()

# Display vocabulary
print("TF-IDF Vocabulary:")
print(tfidf_vocabulary)

print("\nVocabulary Size:")
print(len(tfidf_vocabulary))

# Display matrix shape
print("\nTF-IDF Matrix Shape:")
print(tfidf_matrix.shape)

TF-IDF Vocabulary:
['absolutely' 'acting' 'amazing' 'app' 'arrived' 'awesome' 'bad' 'boring'
 'bought' 'brother' 'buy' 'buying' 'camera' 'check' 'condition' 'contact'
 'customer' 'definitely' 'delivery' 'do' 'download' 'email' 'excellent'
 'expensive' 'fast' 'genuine' 'god' 'good' 'gr' 'great' 'happy' 'help'
 'helpful' 'hour' 'kinda' 'know' 'late' 'latest' 'loove' 'mail' 'movie'
 'my' 'offer' 'oh' 'order' 'ordered' 'perfect' 'phone' 'please' 'pm'
 'possible' 'price' 'product' 'purchase' 'quality' 'query' 'really'
 'reliable' 'response' 'rupee' 'service' 'slow' 'soo' 'soon' 'support'
 'team' 'totally' 'waited' 'worth' 'wow' 'yesterday']

Vocabulary Size:
71

TF-IDF Matrix Shape:
(30, 71)


## Task 8 -> BoW vs TF-IDF Comparison

In [115]:
# Top 10 most frequent words according to BoW
common_words = frequency_df.sort_values(by='frequency', ascending=False).head(10)["word"].tolist()

print(f'Frequent words according to BoW {common_words}')

total_tfidf = np.asarray(tfidf_matrix.sum(axis=0)).flatten()

tfidf_df = pd.DataFrame({
    "word": tfidf_vocabulary,
    "total_tfidf": total_tfidf
    }
)
common_word_tfidf = tfidf_df[tfidf_df['word'].isin(common_words)].sort_values(by='total_tfidf', ascending=False)
common_word_tfidf

Frequent words according to BoW ['product', 'soo', 'delivery', 'good', 'movie', 'bad', 'amazing', 'my', 'service', 'slow']


,word,total_tfidf
52,product,3.708874
27,good,2.117897
62,soo,2.031496
18,delivery,1.762247
40,movie,1.538344
2,amazing,1.129982
60,service,1.013834
61,slow,0.967256
6,bad,0.913350
41,my,0.824186


In [111]:
# High TF-IDF Words
high_tfidf_words = tfidf_df.sort_values(by="total_tfidf",ascending=False)
print("Words with High TF-IDF:")
high_tfidf_words.head(10)

Words with High TF-IDF:


,word,total_tfidf
52,product,3.708874
27,good,2.117897
62,soo,2.031496
18,delivery,1.762247
40,movie,1.538344
30,happy,1.272155
2,amazing,1.129982
28,gr,1.097928
68,worth,1.048991
60,service,1.013834


In [112]:
# Low TF-IDF Words
low_tfidf_words = tfidf_df.sort_values(by="total_tfidf",ascending=True)
print("Words with Low TF-IDF:")
low_tfidf_words.head(10)

Words with Low TF-IDF:


,word,total_tfidf
16,customer,0.415408
32,helpful,0.415408
58,response,0.415408
64,support,0.415408
65,team,0.415408
0,absolutely,0.439380
47,phone,0.439380
29,great,0.463055
34,kinda,0.463055
1,acting,0.463055


## Task 9 -> Vectorizer Parameter Exploration

In [ ]:
# CountVectorizer with max_features=20, vectorizer keeps at most 20 features.
bow_max_features = CountVectorizer(max_features=20)

max_features_matrix = bow_max_features.fit_transform(df['final_processed_text'])

print("\nVocabulary Size with max_features=20")
print(len(bow_max_features.get_feature_names_out()))

print("\nSelected Features:")
print(bow_max_features.get_feature_names_out())


Vocabulary Size with max_features=20
20

Selected Features:
['amazing' 'bad' 'check' 'condition' 'contact' 'delivery' 'fast' 'god'
 'good' 'gr' 'happy' 'movie' 'my' 'oh' 'product' 'quality' 'service'
 'slow' 'soo' 'worth']


In [140]:
# CountVectorizer with min_df = 2, a word needs to occur in at least 2 documents to be retained. 
bow_min_df = CountVectorizer(min_df=2)

bow_min_matrix = bow_min_df.fit_transform(df['final_processed_text'])

print("Vocabulary Size with min_df=2:")
print(len(bow_min_df.get_feature_names_out()))

# CountVectorizer with max_df = 0.8, words appearing in more than 80% of documents can be excluded.
bow_max_df = CountVectorizer(max_df=0.8)

max_df_matrix = bow_max_df.fit_transform(df["final_processed_text"])

print("Vocabulary Size with max_df=0.80:")
print(len(bow_max_df.get_feature_names_out()))


Vocabulary Size with min_df=2:
16
Vocabulary Size with max_df=0.80:
71


## Task 10 -> Conceptual Questions

#### 1. 
#### One-Hot Encoding represents whether a word is present or absent using 1 and 0. 
#### Bag of Words represents the frequency/ of each word in a document.

#### 2. 
#### N-Grams create additional features from sequences of words. Unigrams represent individual words, 
#### while bigrams and trigrams create two-word and three-word combinations. As the number of possible combinations increases, the vocabulary and feature space also increase.

#### 3.
#### TF-IDF can be preferred when we want to give greater importance to words that are more specific to individual documents and reduce the influence of words that occur frequently across many documents. 
#### BoW is useful when raw word frequency is sufficient.

#### 4.
#### Count-based techniques such as One-Hot Encoding and BoW have several limitations.
#### They can produce high-dimensional and sparse vectors, ignore word order, and have limited ability to capture semantic relationships between words.
#### Words with similar meanings are treated as separate features.